# Linear Regression in Python

---

## 1. Introduction

This notebook demonstrates how to fit linear regression models in a Jupyter notebook using standard Python libraries and data from **NHANES**. NHANES is a complex, weighted survey (with strata and clusters) that ordinarily requires survey‑aware methods; in this notebook we ignore the survey design in order to illustrate regression techniques for independent or convenience samples.

We focus on models where **systolic blood pressure (SBP)** is the outcome (dependent) variable — that is, we  predict an individual’s SBP from other subject‑level variables. A domain expert has suggested several plausible predictors: age, body‑mass index (BMI), sex (or gender), and race/ethnicity, since SBP tends to increase with age, is higher among people with greater BMI, and varies across demographic groups.

Linear regression is a natural, easy‑to‑interpret starting point for modeling a quantitative outcome such as SBP. Note that the standard ordinary least squares (OLS) inference — which we use in this notebook — relies on assumptions such as linearity, independence, homoscedasticity, and, for some tests, approximately normal errors.

More complex approaches (e.g., multilevel/mixed‑effects models and marginal/GEE models) will be taught in Week 3; we start with OLS so you have a clear foundation in estimation, interpretation, and diagnostic checks that these later methods build on.

---

## 2. Import Libraries, Load and Prepare the Data
First, we import the standard libraries for data manipulation, visualization, and statistical modeling (e.g., pandas, numpy, matplotlib/seaborn, and statsmodels). 

Then we load the NHANES 2015–2016 data. NHANES includes multiple waves; this notebook uses only the 2015–2016 wave.

As with most datasets, NHANES contains missing values. For simplicity we perform a **complete‑case analysis**: we  drop observations with missing values in any of the key variables used in this notebook. 

> **Note:** Complete‑case analysis reduces the effective sample size and can bias estimates and standard errors unless the missingness mechanism is missing completely at random (MCAR). In practice you should consider alternatives such as single or multiple imputation, or other methods that explicitly model missingness.

After loading and cleaning, we rename columns to make variable names easier to work with (for readability and convenience).

In [48]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

# Read the 2015-2016 wave of NHANES data
nhanes_df = pd.read_csv("./data/nhanes_2015_2016.csv")

# Select columns of interest and drop observations with missing values
cols = ["BPXSY1", "RIDAGEYR", "RIAGENDR", "BMXBMI"]
nhanes_df = nhanes_df[cols].dropna().reset_index(drop=True)

# Rename columns
nhanes_df = nhanes_df.rename(columns={"BPXSY1": "sbp", "RIDAGEYR" : "age", "RIAGENDR": "gender", "BMXBMI": "bmi"})

# Print the first 5 rows
nhanes_df.head()

,sbp,age,gender,bmi
0,128.0,62,1,27.8
1,146.0,53,1,30.8
2,138.0,78,1,28.8
3,132.0,56,2,42.4
4,100.0,42,2,20.3



---

## 3. Fitting a Simple Linear Regression Model
We start with a simple linear regression model that uses a single covariate, age, to predict systolic blood pressure (SBP).

In our dataframe, the variable `sbp` contains the first recorded SBP measurement for each subject and `age` is the subject's age in years. The formula `sbp ~ age` indicates that `sbp` is the response variable and `age` is the single predictor.

In [49]:
model = sm.OLS.from_formula("sbp ~ age", data=nhanes_df)
result = model.fit()
print(result.summary())
print("="*78)

                            OLS Regression Results                            
Dep. Variable:                    sbp   R-squared:                       0.222
Model:                            OLS   Adj. R-squared:                  0.222
Method:                 Least Squares   F-statistic:                     1522.
Date:                Wed, 22 Oct 2025   Prob (F-statistic):          3.09e-293
Time:                        21:09:24   Log-Likelihood:                -22487.
No. Observations:                5347   AIC:                         4.498e+04
Df Residuals:                    5345   BIC:                         4.499e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept    102.4739      0.619    165.512      0.0

### Interpretting Regression Parameters

We will focus on the center section of the OLS output where the header row begins with **coef**. That section shows the estimated regression parameters (the coefficients), their standard errors, t‑statistics, p‑values, and related quantities used to quantify uncertainty. Regression parameters are often called *slopes* or *effects*.
